In [ ]:
# Diffusion model dependencies (TabDDPM + CoDi)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# CoDi: ChaejeongLee/CoDi (_vendor/CoDi)
%pip install -q ForestDiffusion xgboost category-encoders libzero rtdl imbalanced-learn absl-py tensorboardX

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "goggle" / "src"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "CoDi"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_codi


In [ ]:
from ucimlrepo import fetch_ucirepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

concrete = fetch_ucirepo(id=165)
X = concrete.data.features
y = concrete.data.targets
print(concrete.metadata)
print(concrete.variables)

data = pd.concat([X, y], axis=1)
target_col = "Concrete compressive strength"

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


In [ ]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# CoDi
if 'CoDi' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training CoDi...')
        synthetic_codi = train_codi(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['CoDi'] = synthetic_codi.copy()
        print('CoDi: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_codi,
                metadata=train_metadata,
            )
            scores['CoDi'] = quality.get_score()
            print('CoDi:', round(scores['CoDi'], 4))
        else:
            print('CoDi: trained (quality eval skipped)')
    except Exception as e:
        print('CoDi Failed (training/sampling):')
        traceback.print_exc()
    if 'CoDi' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('CoDi: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        pass  # keep continuous regression target as-is

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )


In [ ]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


In [ ]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if not use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [ ]:
print('TRTR (Train Real, Test Real) — 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=True,
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=True,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


In [ ]:
output_file = 'TRTR_TSTR_results_air_quality.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')
